# Visualización interactiva en Jupyter

Este notebook está preparado para ejecutarse en Jupyter Notebook/JupyterLab desde el navegador. El slider cambia el índice `i` y redibuja la figura sin acumular gráficos anteriores.

Los archivos `vacancias.dat` e `is.dat` deben estar en la misma carpeta que este notebook.


In [ ]:
# Si ipywidgets no está instalado, ejecutar una sola vez:
# %pip install ipywidgets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets

from pathlib import Path
from IPython.display import display, clear_output

# Para este caso alcanza con el backend inline: el widget vuelve a dibujar
# la figura cada vez que cambia el slider.
%matplotlib inline


In [ ]:
# Carga de datos
base = Path.cwd()
archivo_vacancias = base / "vacancias.dat"
archivo_is = base / "is.dat"

if not archivo_vacancias.exists():
    raise FileNotFoundError(f"No se encontró: {archivo_vacancias}")
if not archivo_is.exists():
    raise FileNotFoundError(f"No se encontró: {archivo_is}")

data = np.loadtxt(archivo_vacancias)
is_data = np.loadtxt(archivo_is)

interfaz = 50.5
paso = 1

# Evita que el slider permita un índice que no exista en alguno de los arrays.
n_frames = min(data.shape[0], is_data.shape[0])

print("data:", data.shape)
print("is_data:", is_data.shape)
print("número de frames:", n_frames)


In [ ]:
def plot_densidades(i):
    # i llega como entero desde el slider
    i = int(i)

    fig = plt.figure(figsize=(11, 7), constrained_layout=True)
    gs = gridspec.GridSpec(2, 3, figure=fig)

    # ----- Panel superior: densidad de vacancias -----
    ax = fig.add_subplot(gs[0, :])

    y = data[i, 1:-1]
    x = np.arange(1, len(y) + 1)

    ax.bar(x, y)
    ax.set_xlabel("sitio", size=15)
    ax.set_ylabel(r"$\delta_i$", size=15)
    ax.axvline(interfaz, color="red", lw=1, alpha=0.75)
    ax.axvline(len(y) + 0.5, color="red", lw=1, alpha=0.75)
    ax.set_title(f"índice {i} / {n_frames - 1}")

    # ----- Paneles inferiores: V, Is y Q -----
    labels = ["V", "Is", "Q"]

    # Usamos la primera columna de is_data como eje temporal, de modo que
    # la curva y el punto resaltado tengan exactamente el mismo eje x.
    t = is_data[:, 0]

    for j, label in enumerate(labels):
        ax = fig.add_subplot(gs[1, j])
        ax.plot(t, is_data[:, j + 1])
        ax.scatter(t[i], is_data[i, j + 1], color="red", zorder=3)
        ax.set_ylabel(label)
        ax.set_xlabel("t [u.a.]")
        ax.tick_params(axis="x", rotation=45)

    fig.align_labels()
    plt.show()
    plt.close(fig)


In [ ]:
# Widget interactivo
slider_i = widgets.IntSlider(
    value=0,
    min=0,
    max=n_frames - 1,
    step=paso,
    description="i:",
    continuous_update=False,
    layout=widgets.Layout(width="80%")
)

salida = widgets.Output()

def actualizar(change=None):
    with salida:
        clear_output(wait=True)
        plot_densidades(slider_i.value)

slider_i.observe(actualizar, names="value")

display(slider_i, salida)
actualizar()
